## Analyzing LeTRS Results on simulated data

In [3]:
# Importing libraries
import pandas as pd
import os
import glob
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [23]:
# Loading data

# LeTRS Results
names = "/restricted/projectnb/challenge2025/markerte/sgRNAtor/Elm/LeTRS_Output/sgenerate_simdata_v1/*/results/known_junction.tab"
all_files = glob.glob(names)
print(all_files)

LeTRS_Res = pd.concat(
    (pd.read_csv(f, sep="\t", skipfooter=7).assign(Run = re.search("(?<=sgenerate_simdata_v1/).*(?=/results)", f).group()) for f in all_files), 
    ignore_index=True
)

LeTRS_Res[['c_ct', 'c_f_only', 'c_b_only', 'c_both', 'c_al_one']] = LeTRS_Res["cluster_count"].str.extract('(.+)\\((.+),(.+),(.+),(.+)\\)')


print(LeTRS_Res.head(20))


['/restricted/projectnb/challenge2025/markerte/sgRNAtor/Elm/LeTRS_Output/sgenerate_simdata_v1/final_COV_S_+_ORF7a_Dominant_rep2_agregate/results/known_junction.tab', '/restricted/projectnb/challenge2025/markerte/sgRNAtor/Elm/LeTRS_Output/sgenerate_simdata_v1/final_COV_Reverse_Decay_rep3_agregate/results/known_junction.tab', '/restricted/projectnb/challenge2025/markerte/sgRNAtor/Elm/LeTRS_Output/sgenerate_simdata_v1/final_COV_M_+_ORF8_Dominant_rep2_agregate/results/known_junction.tab', '/restricted/projectnb/challenge2025/markerte/sgRNAtor/Elm/LeTRS_Output/sgenerate_simdata_v1/final_COV_M_+_ORF8_Dominant_rep1_agregate/results/known_junction.tab', '/restricted/projectnb/challenge2025/markerte/sgRNAtor/Elm/LeTRS_Output/sgenerate_simdata_v1/final_COV_Exponential_Decay_rep1_agregate/results/known_junction.tab', '/restricted/projectnb/challenge2025/markerte/sgRNAtor/Elm/LeTRS_Output/sgenerate_simdata_v1/final_COV_Sparse_1_rep2_agregate/results/known_junction.tab', '/restricted/projectnb/chal

/scratch/416971.1.ood/ipykernel_1289661/3106937991.py:9: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  (pd.read_csv(f, sep="\t", skipfooter=7).assign(Run = re.search("(?<=sgenerate_simdata_v1/).*(?=/results)", f).group()) for f in all_files),
/scratch/416971.1.ood/ipykernel_1289661/3106937991.py:9: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  (pd.read_csv(f, sep="\t", skipfooter=7).assign(Run = re.search("(?<=sgenerate_simdata_v1/).*(?=/results)", f).group()) for f in all_files),
/scratch/416971.1.ood/ipykernel_1289661/3106937991.py:9: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  (pd.read_csv(f, sep="\t", skipfooter=7).assign(Run = re.sear

In [26]:
# sgENERATE "ground truth"

sgen_GT_dict = {"Run": [], "subgenome": [], "sgen_count": []}

for run in LeTRS_Res['Run'].unique():
    f = open(f"/restricted/projectnb/challenge2025/Data/sgenerate_simdata_v1/{run[:-8]}proportion.txt", "r")
    for l in f.readlines():
        sgen_GT_dict["Run"].append(run)
        sgen_GT_dict["subgenome"].append(l.split()[0])
        sgen_GT_dict["sgen_count"].append(l.split()[1])

sgen_gt = pd.DataFrame(sgen_GT_dict)

print(sgen_gt.head())

sgen_gt.to_csv("sgen_groundtruth_ORFs.csv", index = False)

                                          Run subgenome sgen_count
0  final_COV_S_+_ORF7a_Dominant_rep2_agregate         S       1249
1  final_COV_S_+_ORF7a_Dominant_rep2_agregate     ORF3a        208
2  final_COV_S_+_ORF7a_Dominant_rep2_agregate         E        208
3  final_COV_S_+_ORF7a_Dominant_rep2_agregate         M        208
4  final_COV_S_+_ORF7a_Dominant_rep2_agregate      ORF6        208


In [27]:
merged = pd.merge(sgen_gt, LeTRS_Res, on=["Run", "subgenome"],how = "inner")

merged.head()

,Run,subgenome,sgen_count,ref_leader_end,peak_leader_end,ref_TRS_start,peak_TRS_start,peak_count,peak_normalized_count,cluster_count,cluster_normalized_count,c_ct,c_f_only,c_b_only,c_both,c_al_one
0,final_COV_S_+_ORF7a_Dominant_rep2_agregate,S,1249,65,66,21552,21552,"399(362,0,0,362)","2331.79(101230.43,0.00,0.00,3105.35)","445(401,0,0,401)","2600.62(112136.47,0.00,0.00,3439.90)",445,401,0,0,401
1,final_COV_S_+_ORF7a_Dominant_rep2_agregate,ORF3a,208,69,69,25385,25385,"79(77,0,0,77)","461.68(21532.44,0.00,0.00,660.53)","81(79,0,0,79)","473.37(22091.72,0.00,0.00,677.69)",81,79,0,0,79
2,final_COV_S_+_ORF7a_Dominant_rep2_agregate,E,208,69,69,26237,26237,"65(61,0,0,61)","379.87(17058.17,0.00,0.00,523.28)","65(61,0,0,61)","379.87(17058.17,0.00,0.00,523.28)",65,61,0,0,61
3,final_COV_S_+_ORF7a_Dominant_rep2_agregate,M,208,65,65,26469,26469,"75(71,0,0,71)","438.31(19854.59,0.00,0.00,609.06)","84(80,0,0,80)","490.90(22371.36,0.00,0.00,686.27)",84,80,0,0,80
4,final_COV_S_+_ORF7a_Dominant_rep2_agregate,ORF6,208,69,69,27041,27041,"59(57,0,0,57)","344.80(15939.60,0.00,0.00,488.96)","59(57,0,0,57)","344.80(15939.60,0.00,0.00,488.96)",59,57,0,0,57


In [ ]:
grid = sns.FacetGrid() # do more here